The code DAGS i created to practice, using local files 

In [ ]:
from datetime import datetime 
import os
import json
import random

from airflow.providers.standard.operators.python import PythonOperator
from airflow.sdk import dag, task, get_current_context, DAG


def say_hello():
     print("Hello from Apache Airflow!")


with DAG(
    "my_first_dag",
    start_date = datetime(2026,1,1),
    schedule = None,
    catchup= False ,
    ) as dag:

    t1 = PythonOperator(
        task_id = "greet_user" ,
        python_callable = say_hello,
    )


output_path = "/Users/kamari/Documents/project_info/dags"

In [ ]:
from datetime import datetime 
from pyspark.sql import SparkSession, functions as f 
import os
from airflow.providers.standard.operators.python import PythonOperator
from airflow.sdk import dag, task, get_current_context, DAG

input_path = "/Users/kamari/Documents/project_info/employees.csv"
output_path = "/Users/kamari/Documents/project_info/employees.parquet"


def extract(spark, input_path):
    df = spark.read.csv(input_path, header=True, inferSchema=True)
    return(df)

def transform(df, bad_column): 
   new_df = df.drop(bad_column)
   return new_df

def load(df, output):
    df.write.mode('overwrite').parquet(output)
    
def run_etl():
    spark = (SparkSession\
        .builder\
        .appName("ETL Spark Practice")\
        .getOrCreate()
    )
    try:
        df = extract(spark ,input_path)
        
        new_df = transform(df, "COMMISSION_PCT")
        
        load(new_df, output_path)
        
    finally:
        spark.stop()
    

with DAG(
    
    "etl_pipeline",
    start_date= datetime(2026, 1, 1),
    schedule= None ,
    catchup= False , 
) as dag:

        
    t1 = PythonOperator(
        task_id = 'run_etl',
        python_callable= run_etl
    )
    